<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week06-smart-antennas.ipynb)

# Week 6 — Smart Antennas
**ECE 512 Wireless Communications** · companion notebook to the Week 6 lecture

This notebook lets you experiment with the ideas from the slides:

1. Spatial sampling: what differs from one element to the next
2. The steering vector $\mathbf{a}(\theta)$ for ULA and UCA geometries
3. Spatial aliasing and grating lobes ($d > \lambda/2$)
4. Signal model and the spatial covariance matrix $\mathbf{R} = \mathbf{G}\mathbf{P}\mathbf{G}^H + \sigma^2\mathbf{I}$
5. Uplink direction of arrival: Bartlett, Capon (MVDR) and MUSIC
6. Downlink beamforming: dominant DOA, pseudoinverse, spatial-signature methods
7. Downlink SINR with multiple users
8. Why array calibration matters

Conventions (as in the lecture): element positions are measured in wavelengths, so $k = 2\pi/\lambda$;
$\theta$ is measured from the $x$ axis (for a ULA on the $x$ axis, broadside is $\theta = 90^\circ$).

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Checkbox
from ece512 import arrays
from ece512.units import db_to_linear, linear_to_db

DEG = np.pi / 180
theta_ula = np.linspace(0, np.pi, 1801)          # a ULA only "sees" cos(theta): scan 0..180 deg
theta_all = np.linspace(0, 2 * np.pi, 3601)      # full circle for the UCA
rng = np.random.default_rng(512)

## 1. Spatial sampling: what differs from one element to the next?
A narrowband plane wave from direction $\theta$ produces, after downconversion, the element output
$$d_i(t) = s(t)\exp\!\left(-jk\left(x_i\cos\theta + y_i\sin\theta\right)\right).$$
Every element sees the *same* $s(t)$; only the **phase** differs, because the electrical path length
$x_i\cos\theta + y_i\sin\theta$ differs (amplitude differences across a tower-mounted array are small).
For a ULA with spacing $d$ the phase advances by $kd\cos\theta$ per element — a *spatial frequency* that
encodes the direction.

In [ ]:
M, d = 8, 0.5
x, _ = arrays.ula_positions(M, d)
xx = np.linspace(0, x[-1], 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for th_deg in [90, 60, 30, 0]:
    a = arrays.ula_steering(th_deg * DEG, M, d)
    assert np.allclose(np.angle(a), np.angle(np.exp(-1j * 2 * np.pi * x * np.cos(th_deg * DEG))))
    axes[0].plot(np.arange(1, M + 1), -360 * x * np.cos(th_deg * DEG), "o-", label=fr"$\theta$ = {th_deg}°")
for th_deg, col in [(60, "tab:orange"), (30, "tab:green")]:
    a = arrays.ula_steering(th_deg * DEG, M, d)
    axes[1].plot(xx, np.cos(2 * np.pi * xx * np.cos(th_deg * DEG)), color=col, alpha=0.4)
    axes[1].plot(x, a.real, "o", color=col, label=fr"$\theta$ = {th_deg}°: {d * np.cos(th_deg * DEG):.2f} cycles/element")
axes[0].set_xlabel("element index i"); axes[0].set_ylabel(r"phase $-kx_i\cos\theta$ (deg)")
axes[0].set_title(r"Phase slope $-kd\cos\theta$ per element ($d=\lambda/2$)")
axes[1].set_xlabel(r"element position $x_i$ (wavelengths)"); axes[1].set_ylabel(r"Re$\{a_i(\theta)\}$")
axes[1].set_title("Elements sample a spatial sinusoid"); axes[1].set_ylim(-1.1, 1.5)
axes[0].legend(fontsize=8, loc="lower left"); axes[1].legend(fontsize=8, loc="upper right", ncol=2)
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. The steering vector: ULA and UCA
Stacking the element responses gives the **steering vector** — the "spatial impulse response" of the array:
$$a_i(\theta) = \exp\!\left(-jk\left(x_i\cos\theta + y_i\sin\theta\right)\right),\qquad i = 1,\ldots,M$$
* ULA: $x_i = (i-1)d$, $y_i = 0$, so $\mathbf{a}_{\mathrm{ULA}}(\theta) = \left[1,\ e^{-jkd\cos\theta},\ \ldots,\ e^{-jk(M-1)d\cos\theta}\right]^T$
* UCA: $x_i = R\cos\!\left(\tfrac{2\pi(i-1)}{M}\right)$, $y_i = R\sin\!\left(\tfrac{2\pi(i-1)}{M}\right)$

If we combine the element outputs with weights $\mathbf{w}$, $y(t) = \mathbf{w}^H\mathbf{x}(t)$, a unit plane wave from $\theta$
produces $\mathbf{w}^H\mathbf{a}(\theta)$. Choosing $\mathbf{w} = \mathbf{a}(\theta_0)$ "steers" the beam to $\theta_0$; the
pattern $|\mathbf{a}^H(\theta_0)\mathbf{a}(\theta)|^2$ peaks at $M^2$ there.

Try: the ULA pattern is **mirror-symmetric** about the array axis ($\theta$ and $-\theta$ give the same
$\cos\theta$) — a ULA cannot tell front from back. The UCA has no such ambiguity. For the UCA the slider sets
the spacing between *adjacent* elements.

In [ ]:
def show_pattern(geometry="ULA", M=8, spacing=0.5, steer_deg=60.0):
    if geometry == "ULA":
        x, y = arrays.ula_positions(M, spacing)
    else:
        x, y = arrays.uca_positions(M, arrays.uca_radius_for_spacing(M, spacing))
    w = arrays.steering_vector(steer_deg * DEG, x, y)
    A = arrays.steering_vector(theta_all, x, y)
    p_db = arrays.normalized_pattern_db(w, A, floor_db=-40)

    fig = plt.figure(figsize=(11, 4.4))
    ax0 = fig.add_subplot(1, 2, 1)
    ax0.plot(x, y, "ko"); ax0.set_aspect("equal")
    r = max(np.abs(x).max(), np.abs(y).max()) + 0.6
    ax0.annotate("", xy=(np.mean(x) + 0.9 * r * np.cos(steer_deg * DEG), np.mean(y) + 0.9 * r * np.sin(steer_deg * DEG)),
                 xytext=(np.mean(x), np.mean(y)), arrowprops=dict(arrowstyle="->", color="tab:red", lw=2))
    ax0.set_xlim(-r + np.mean(x), r + np.mean(x)); ax0.set_ylim(-r, r)
    ax0.set_xlabel(r"$x$ (wavelengths)"); ax0.set_ylabel(r"$y$ (wavelengths)")
    ax0.set_title(f"{geometry} geometry, M={M}, steered to {steer_deg:.0f}°"); ax0.grid(alpha=0.3)

    ax1 = fig.add_subplot(1, 2, 2, projection="polar")
    ax1.plot(theta_all, p_db + 40, color="tab:blue")
    ax1.set_rticks([0, 10, 20, 30, 40]); ax1.set_yticklabels(["-40", "-30", "-20", "-10", "0 dB"], fontsize=7)
    ax1.set_rlim(0, 41); ax1.set_rlabel_position(250)
    ax1.set_title(r"$|\mathbf{a}^H(\theta_0)\mathbf{a}(\theta)|^2$ (dB, normalized)", pad=14)
    plt.tight_layout(); plt.show()

interact(show_pattern, geometry=Dropdown(options=["ULA", "UCA"], value="ULA"),
         M=IntSlider(value=8, min=2, max=24),
         spacing=FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description="d / λ"),
         steer_deg=FloatSlider(value=60, min=0, max=360, step=5, description="θ₀ (deg)"));

**Beamwidth check.** For a ULA steered to broadside the pattern is the Dirichlet kernel
$\left|\frac{\sin(M\psi/2)}{\sin(\psi/2)}\right|^2$ with $\psi = kd(\cos\theta - \cos\theta_0)$, so the first nulls sit
at $\cos\theta - \cos\theta_0 = \pm 1/(Md)$. Simulation (numerical pattern) vs. theory:

In [ ]:
print(f"{'M':>3} {'d/λ':>5} {'first null (theory)':>20} {'first null (numerical)':>23}")
for M in [4, 8, 16]:
    for d in [0.5, 0.3]:
        p = arrays.array_pattern(arrays.ula_steering(90 * DEG, M, d), arrays.ula_steering(theta_ula, M, d))
        i0 = np.argmin(np.abs(theta_ula - 90 * DEG))
        k = i0 + np.argmax(np.diff(p[i0:]) > 0)      # first local minimum right of broadside
        theory = np.arccos(-1 / (M * d)) / DEG
        print(f"{M:>3} {d:>5} {theory:>17.2f} deg {theta_ula[k] / DEG:>20.2f} deg")

## 3. Spatial aliasing and grating lobes
Spatial sampling is analogous to temporal sampling. The ULA samples the spatial sinusoid of Section 1 every $d$;
the "spatial frequency" $d\cos\theta$ (cycles per element) is unambiguous only if it stays within $[-\tfrac12, \tfrac12]$.
When $d > \lambda/2$ two different directions can produce the *same* steering vector, and a beam steered to $\theta_0$
has full-gain **grating lobes** wherever
$$kd(\cos\theta - \cos\theta_0) = 2\pi m \iff \cos\theta = \cos\theta_0 + \frac{m}{d},\quad m = \pm1, \pm2, \ldots$$
(The spacing that avoids grating lobes over the whole scan range $0^\circ$–$180^\circ$ is $d < \lambda/2$.)
Dashed lines below mark the theoretical lobe directions.

In [ ]:
def show_grating(d=1.0, steer_deg=70.0, M=8):
    A = arrays.ula_steering(theta_ula, M, d)
    p_db = arrays.normalized_pattern_db(arrays.ula_steering(steer_deg * DEG, M, d), A, floor_db=-40)
    lobes = arrays.ula_grating_lobes(steer_deg * DEG, d) / DEG
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.plot(theta_ula / DEG, p_db, label=fr"pattern, $d$ = {d:.2f}$\lambda$")
    labelled = False
    for lb in lobes:
        if abs(lb - steer_deg) < 1e-6:
            ax.axvline(lb, color="tab:green", alpha=0.8, label=r"main lobe $\theta_0$")
        else:
            ax.axvline(lb, color="tab:red", ls="--", alpha=0.8, label=None if labelled else "grating lobe (theory)")
            labelled = True
    ax.set_xlim(0, 180); ax.set_ylim(-40, 3)
    ax.set_xlabel(r"$\theta$ (deg)"); ax.set_ylabel("normalized gain (dB)")
    n_gl = len(lobes) - 1
    ax.set_title(f"ULA M={M} steered to {steer_deg:.0f}°: {n_gl} grating lobe(s)")
    ax.grid(alpha=0.3); ax.legend(loc="lower right", fontsize=8); plt.show()
    # identical steering vectors => a DOA algorithm cannot tell these directions apart
    for lb in lobes:
        if abs(lb - steer_deg) > 1e-6:
            c = abs(np.vdot(arrays.ula_steering(steer_deg * DEG, M, d), arrays.ula_steering(lb * DEG, M, d))) / M
            print(f"|a^H({steer_deg:.0f}°) a({lb:.1f}°)| / M = {c:.4f}  -> indistinguishable directions")

interact(show_grating, d=FloatSlider(value=1.0, min=0.25, max=2.5, step=0.05, description="d / λ"),
         steer_deg=FloatSlider(value=70, min=0, max=180, step=5, description="θ₀ (deg)"),
         M=IntSlider(value=8, min=2, max=24));

## 4. Signal model and the spatial covariance matrix
One user with $q$ multipath components gives $\mathbf{x}(t) = \sum_{i=1}^{q}\alpha_i\,\mathbf{a}(\theta_i)\,s(t) + \mathbf{n}(t)$,
summarized by the **spatial signature** $\mathbf{g} = \sum_i \alpha_i\mathbf{a}(\theta_i)$. With $L$ users,
$\mathbf{x}(t) = \mathbf{G}\mathbf{s}(t) + \mathbf{n}(t)$, $\mathbf{G} = [\mathbf{g}_1\ \cdots\ \mathbf{g}_L]$, and
$$\mathbf{R} = \mathrm{E}\!\left[\mathbf{x}\mathbf{x}^H\right] = \mathbf{G}\mathbf{P}\mathbf{G}^H + \sigma^2\mathbf{I}
 = \mathbf{U}_s\mathbf{\Lambda}_s\mathbf{U}_s^H + \mathbf{U}_n\mathbf{\Lambda}_n\mathbf{U}_n^H,
\qquad \hat{\mathbf{R}} = \frac{1}{N}\sum_{i=1}^{N}\mathbf{x}(iT_s)\,\mathbf{x}^H(iT_s).$$
Because $\mathbf{G}\mathbf{P}\mathbf{G}^H$ has rank $L$, the exact $\mathbf{R}$ has $L$ "signal" eigenvalues above
$\sigma^2$ and $M-L$ noise eigenvalues **exactly equal to** $\sigma^2$. With finitely many snapshots the noise eigenvalues
of $\hat{\mathbf{R}}$ spread out around $\sigma^2$. Here each user is a single path ($\mathbf{g}_l = \mathbf{a}(\theta_l)$),
users are equally spaced in angle, and SNR $= P/\sigma^2$ per user per element.

In [ ]:
def show_eigs(L=2, snr_db=10.0, N=50, M=8):
    sigma2 = 1.0
    P = db_to_linear(snr_db) * sigma2
    doas = np.linspace(40, 140, L + 2)[1:-1] * DEG
    G = arrays.ula_steering(doas, M, 0.5)
    R = arrays.covariance(G, [P] * L, sigma2)
    X = arrays.simulate_snapshots(G, [P] * L, sigma2, N, rng=rng)
    lam, Us, Un = arrays.subspaces(R, L)
    lam_hat = arrays.eig_desc(arrays.sample_covariance(X))[0]

    fig, ax = plt.subplots(figsize=(8, 3.8))
    k = np.arange(1, M + 1)
    ax.plot(k[:L], linear_to_db(lam[:L]), "s", ms=10, color="tab:blue", label=r"exact $\mathbf{R}$: signal")
    ax.plot(k[L:], linear_to_db(lam[L:]), "s", ms=10, color="tab:gray", label=r"exact $\mathbf{R}$: noise ($=\sigma^2$)")
    ax.plot(k, linear_to_db(lam_hat), "o", ms=7, color="tab:orange", mec="k", label=fr"sample $\hat{{\mathbf{{R}}}}$ (N={N})")
    ax.axhline(linear_to_db(sigma2), color="k", ls="--", lw=1)
    ax.axvline(L + 0.5, color="tab:red", ls=":", lw=1.5)
    ax.set_ylim(min(-5, linear_to_db(lam_hat.min()) - 2), linear_to_db(max(lam[0], lam_hat[0])) + 4)
    ax.annotate("signal | noise subspace", xy=(L + 0.5, 0.55), xycoords=("data", "axes fraction"),
                ha="center", va="top", color="tab:red", fontsize=9, backgroundcolor="w")
    ax.set_xticks(k); ax.set_xlabel("eigenvalue index (sorted high → low)")
    ax.set_ylabel("eigenvalue (dB)"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="center right")
    ax.set_title(f"M={M}, L={L} sources at " + ", ".join(f"{t / DEG:.0f}°" for t in doas) + f", SNR={snr_db:.0f} dB")
    plt.show()
    print("orthogonality ||U_n^H a(theta_l)||:", np.round(np.linalg.norm(Un.conj().T @ G, axis=0), 12))
    print("MDL estimate of the number of sources from R_hat:", arrays.mdl_order(lam_hat, N))

interact(show_eigs, L=IntSlider(value=2, min=1, max=5), snr_db=FloatSlider(value=10, min=-15, max=30, step=1, description="SNR (dB)"),
         N=IntSlider(value=50, min=5, max=1000, step=5, description="snapshots N"), M=IntSlider(value=8, min=6, max=16));

**Theory check.** For one source ($L=1$), $\mathbf{R} = P\,\mathbf{a}\mathbf{a}^H + \sigma^2\mathbf{I}$ and
$\mathbf{a}^H\mathbf{a} = M$, so the signal eigenvalue is $\lambda_1 = MP + \sigma^2$: the array gain $M$ lifts the signal
above the noise floor even when the per-element SNR is below 0 dB.

In [ ]:
sigma2, P, N = 1.0, db_to_linear(-5), 2000
print(f"{'M':>3} {'lambda_1 theory (dB)':>21} {'lambda_1 from R_hat (dB)':>25}")
for M in [4, 8, 16, 32]:
    a = arrays.ula_steering(75 * DEG, M, 0.5)
    X = arrays.simulate_snapshots(a[:, None], [P], sigma2, N, rng=rng)
    print(f"{M:>3} {linear_to_db(M * P + sigma2):>21.2f} {linear_to_db(arrays.eig_desc(arrays.sample_covariance(X))[0][0]):>25.2f}")

## 5. Uplink direction of arrival: Bartlett, Capon and MUSIC
All three algorithms scan $\theta$ and evaluate a "spatial spectrum" from the *same* $\hat{\mathbf{R}}$:

| method | spatial spectrum | idea |
|---|---|---|
| Bartlett (lecture) | $P_{\mathrm{BF}}(\theta) = \dfrac{\mathbf{a}^H(\theta)\hat{\mathbf{R}}\mathbf{a}(\theta)}{\mathbf{a}^H(\theta)\mathbf{a}(\theta)}$ | output power of $\mathbf{w} = \mathbf{a}(\theta)/\lVert\mathbf{a}\rVert$ |
| Capon / MVDR (extra) | $P_{\mathrm{MVDR}}(\theta) = \dfrac{1}{\mathbf{a}^H(\theta)\hat{\mathbf{R}}^{-1}\mathbf{a}(\theta)}$ | min. output power with unit gain toward $\theta$ |
| MUSIC (lecture) | $P_{\mathrm{MUSIC}}(\theta) = \dfrac{\mathbf{a}^H(\theta)\mathbf{a}(\theta)}{\mathbf{a}^H(\theta)\mathbf{\Pi}_n\mathbf{a}(\theta)}$, $\mathbf{\Pi}_n = \mathbf{U}_n\mathbf{U}_n^H$ | true steering vectors are orthogonal to the noise subspace |

$\hat{\theta}_{\mathrm{DOA\text{-}Estimated}} = \arg\max_\theta P(\theta)$ (the $L$ largest peaks). Bartlett cannot resolve
sources closer than about one beamwidth ($\approx 1/(Md)$ rad at broadside, $14^\circ$ for $M=8$, $d=\lambda/2$);
Capon and especially MUSIC do much better at high SNR and with enough snapshots.

Toggle **coherent** to make the two "sources" multipath copies of *one* user signal (the lecture's
$\sum_i\alpha_i\mathbf{a}(\theta_i)s(t)$). Then $\mathbf{G}\mathbf{P}\mathbf{G}^H$ has rank 1, the signal subspace no longer
contains both steering vectors, and MUSIC/Capon break down (remedies such as spatial smoothing are beyond this lecture).

In [ ]:
def doa_spectra(R, L, A):
    return {"Bartlett": arrays.bartlett_spectrum(R, A),
            "Capon (MVDR)": arrays.capon_spectrum(R, A),
            "MUSIC": arrays.music_spectrum(R, A, L)}

def show_doa(separation_deg=10.0, snr_db=10.0, N=100, M=8, coherent=False):
    center = 90.0
    doas = np.array([center - separation_deg / 2, center + separation_deg / 2]) * DEG
    A_true = arrays.ula_steering(doas, M, 0.5)
    sigma2, P = 1.0, db_to_linear(snr_db)
    if coherent:                                           # one user, two paths: g = a(th1) + a(th2) e^{j phi}
        G = arrays.spatial_signature([1, np.exp(1j * 0.7)], A_true)[:, None]
        X = arrays.simulate_snapshots(G, [P], sigma2, N, rng=rng)
    else:
        X = arrays.simulate_snapshots(A_true, [P, P], sigma2, N, rng=rng)
    R_hat = arrays.sample_covariance(X)
    A = arrays.ula_steering(theta_ula, M, 0.5)
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, Pth in doa_spectra(R_hat, 2, A).items():
        est = arrays.find_peaks(Pth, theta_ula, 2)
        ok = arrays.resolved(est, doas, max(1.0, separation_deg / 4) * DEG)
        ax.plot(theta_ula / DEG, linear_to_db(Pth / Pth.max()),
                label=f"{name}: " + ", ".join(f"{e / DEG:.1f}°" for e in est) + ("  ✓" if ok else "  ✗"))
    for t in doas:
        ax.axvline(t / DEG, color="k", ls="--", lw=1)
    ax.set_xlim(40, 140); ax.set_ylim(-40, 2)
    ax.set_xlabel(r"$\theta$ (deg)"); ax.set_ylabel("normalized spectrum (dB)")
    ax.set_title(f"M={M}, true DOAs (dashed) {doas[0] / DEG:.1f}° & {doas[1] / DEG:.1f}°, SNR={snr_db:.0f} dB, N={N}"
                 + (", coherent" if coherent else ""))
    ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower center"); plt.show()

interact(show_doa, separation_deg=FloatSlider(value=10, min=1, max=40, step=0.5, description="Δθ (deg)"),
         snr_db=FloatSlider(value=10, min=-10, max=40, step=1, description="SNR (dB)"),
         N=IntSlider(value=100, min=4, max=1000, step=4, description="snapshots N"),
         M=IntSlider(value=8, min=3, max=24), coherent=Checkbox(value=False, description="coherent"));

**Resolution vs. angular separation (Monte Carlo).** For each separation $\Delta\theta$ we draw fresh snapshots
many times and count a trial as *resolved* when the two largest peaks lie within $\max(1^\circ, \Delta\theta/4)$ of the
true DOAs. The dotted line is the Rayleigh limit (first null of the beam, $\arccos(-1/(Md)) - 90^\circ$).

In [ ]:
def resolution_curve(seps_deg, snr_db, N, M=8, trials=100):
    A = arrays.ula_steering(np.linspace(60, 120, 1201) * DEG, M, 0.5)
    th = np.linspace(60, 120, 1201) * DEG
    out = {k: [] for k in ["Bartlett", "Capon (MVDR)", "MUSIC"]}
    for sep in seps_deg:
        doas = np.array([90 - sep / 2, 90 + sep / 2]) * DEG
        G = arrays.ula_steering(doas, M, 0.5)
        hits = dict.fromkeys(out, 0)
        for _ in range(trials):
            R_hat = arrays.sample_covariance(arrays.simulate_snapshots(G, [db_to_linear(snr_db)] * 2, 1.0, N, rng=rng))
            for name, Pth in doa_spectra(R_hat, 2, A).items():
                hits[name] += arrays.resolved(arrays.find_peaks(Pth, th, 2), doas, max(1.0, sep / 4) * DEG)
        for name in out:
            out[name].append(hits[name] / trials)
    return out

seps = np.arange(1, 21, 1.0)
M = 8
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, (snr_db, N) in zip(axes, [(10, 100), (25, 100)]):
    for name, prob in resolution_curve(seps, snr_db, N, M).items():
        ax.plot(seps, prob, "o-", ms=3, label=name)
    ax.axvline(np.arccos(-1 / (M * 0.5)) / DEG - 90, color="k", ls=":", label="Rayleigh limit")
    ax.set_xlabel(r"angular separation $\Delta\theta$ (deg)"); ax.set_title(f"M={M}, SNR={snr_db} dB, N={N}")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Pr(resolved)"); axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

## 6. Downlink beamforming weights
With the DOAs (or spatial signatures) learned on the uplink, the base station chooses a weight vector per user.
We use the same pattern convention as the uplink, $|\mathbf{w}^H\mathbf{a}(\theta)|^2$ (by reciprocity this is the
radiated pattern when element $i$ is fed $w_i^*$). The lecture's four methods, for desired user 1 and co-channel
user 2, each with a dominant path ("DOM") and one multipath ("MULTI"):

| method | weight | effect |
|---|---|---|
| Dominant DOA | $\mathbf{w} = \mathbf{a}(\theta_{\mathrm{DOM}_1})$ | beam toward user 1's strongest path |
| Pseudoinverse DOA | $\mathbf{w} = \left[\mathrm{row}(\mathrm{pinv}(\mathbf{A}), 1)\right]^H$, $\mathbf{A} = [\mathbf{a}(\theta_{\mathrm{DOM}_1})\ \mathbf{a}(\theta_{\mathrm{MULTI}_1})\ \mathbf{a}(\theta_{\mathrm{DOM}_2})\ \mathbf{a}(\theta_{\mathrm{MULTI}_2})]$ | unit gain on $\theta_{\mathrm{DOM}_1}$, nulls on the other three |
| Spatial signature | $\mathbf{w} = \mathbf{g}_1$ | beams on all of user 1's paths |
| Pseudoinverse spatial signature | $\mathbf{w} = \left[\mathrm{row}(\mathrm{pinv}(\mathbf{A}), 1)\right]^H$, $\mathbf{A} = [\mathbf{g}_1\ \mathbf{g}_2]$ | $\mathbf{w}^H\mathbf{g}_1 = 1$, $\mathbf{w}^H\mathbf{g}_2 = 0$ |

Why the pseudoinverse works: $\mathbf{A}^\dagger\mathbf{A} = \mathbf{I}$, so the first row $\mathbf{r}$ of $\mathbf{A}^\dagger$ satisfies
$\mathbf{r}\,\mathbf{a}_1 = 1$ and $\mathbf{r}\,\mathbf{a}_j = 0$ for $j \neq 1$; with $\mathbf{w} = \mathbf{r}^H$ this is exactly
$\mathbf{w}^H\mathbf{a}_j = \delta_{j1}$. Solid lines: user 1 paths; dashed: user 2 paths (as in the slides).

In [ ]:
def bf_scenario(th_dom1, th_multi1, th_dom2, th_multi2, M, alpha_multi=0.6):
    """Steering vectors and spatial signatures for the two-user scenario of the slides."""
    a = lambda t: arrays.ula_steering(t * DEG, M, 0.5)
    A_paths = np.column_stack([a(th_dom1), a(th_multi1), a(th_dom2), a(th_multi2)])
    g1 = arrays.spatial_signature([1.0, alpha_multi * np.exp(1j * 1.1)], A_paths[:, :2])
    g2 = arrays.spatial_signature([1.0, alpha_multi * np.exp(-1j * 0.4)], A_paths[:, 2:])
    return A_paths, g1, g2

def bf_weights(A_paths, g1, g2):
    return {"Dominant DOA": arrays.dominant_doa_weights(A_paths[:, 0]),
            "Pseudoinverse DOA": arrays.pinv_weights(A_paths),
            "Spatial signature": g1,
            "Pseudoinverse spatial signature": arrays.pinv_weights(np.column_stack([g1, g2]))}

def show_beamforming(th_dom2=110.0, th_multi2=35.0, M=8):
    th_dom1, th_multi1 = 60.0, 140.0
    A_paths, g1, g2 = bf_scenario(th_dom1, th_multi1, th_dom2, th_multi2, M)
    A = arrays.ula_steering(theta_ula, M, 0.5)
    fig, axes = plt.subplots(2, 2, figsize=(12, 6.2), sharex=True, sharey=True)
    print(f"{'method':<32} {'|w^H g1|^2/||w||^2':>19} {'|w^H g2|^2/||w||^2':>19}")
    for ax, (name, w) in zip(axes.flat, bf_weights(A_paths, g1, g2).items()):
        w = arrays.unit_norm(w)
        ax.plot(theta_ula / DEG, arrays.normalized_pattern_db(w, A, floor_db=-50), color="tab:blue")
        for t in (th_dom1, th_multi1):
            ax.axvline(t, color="tab:green", lw=1.5)
        for t in (th_dom2, th_multi2):
            ax.axvline(t, color="tab:red", ls="--", lw=1.5)
        ax.set_title(name, fontsize=10); ax.grid(alpha=0.3); ax.set_ylim(-50, 3)
        print(f"{name:<32} {abs(np.vdot(w, g1)) ** 2:>19.3f} {abs(np.vdot(w, g2)) ** 2:>19.2e}")
    for ax in axes[1]: ax.set_xlabel(r"$\theta$ (deg)")
    for ax in axes[:, 0]: ax.set_ylabel(r"$|\mathbf{w}^H\mathbf{a}(\theta)|^2$ (dB, norm.)")
    fig.suptitle(fr"M={M}: user 1 (solid green) DOM {th_dom1:.0f}°, MULTI {th_multi1:.0f}°;"
                 fr"  user 2 (dashed red) DOM {th_dom2:.0f}°, MULTI {th_multi2:.0f}°", fontsize=10)
    plt.tight_layout(); plt.show()

interact(show_beamforming, th_dom2=FloatSlider(value=110, min=5, max=175, step=1, description="θ_DOM2"),
         th_multi2=FloatSlider(value=35, min=5, max=175, step=1, description="θ_MULTI2"),
         M=IntSlider(value=8, min=4, max=24));

Things to notice: the spatial-signature beam puts power on *both* of user 1's paths (their gains add coherently in
$|\mathbf{w}^H\mathbf{g}_1|^2$), the pseudoinverse DOA beam nulls user 1's own multipath too (it treats it like an
interferer), and the pseudoinverse spatial-signature beam need not have true nulls in any direction — it only needs
user 2's *paths to cancel in sum*, which is the weakest constraint and so keeps the most gain toward user 1.
Move $\theta_{\mathrm{DOM}_2}$ close to $\theta_{\mathrm{DOM}_1} = 60^\circ$: forcing a null right next to the beam costs a lot of gain.

## 7. Downlink SINR with several users
Now $L$ co-channel users are served simultaneously (SDMA), each with its own unit-norm beam $\mathbf{w}_l$ and power $P$.
User $l$, with spatial signature $\mathbf{g}_l$, receives
$$\mathrm{SINR}_l = \frac{P\,|\mathbf{w}_l^H\mathbf{g}_l|^2}{\sum_{k\neq l}P\,|\mathbf{w}_k^H\mathbf{g}_l|^2 + \sigma^2}.$$
Every user has a dominant path ($\alpha = 1$) plus one multipath with random phase and amplitude $0.6$, at random
angles in $[20^\circ, 160^\circ]$. The upper bound (no interference, beam matched to $\mathbf{g}_l$) is
$P\lVert\mathbf{g}_l\rVert^2/\sigma^2$. Curves show the **median** over random geometries.

In [ ]:
def sinr_vs_snr(M=8, L=2, trials=300, snrs_db=np.arange(-10, 31, 2.5)):
    gains = {k: [] for k in ["Dominant DOA", "Pseudoinverse DOA", "Spatial signature", "Pseudoinverse spatial signature"]}
    bound = []
    for _ in range(trials):
        th = rng.uniform(20, 160, size=(L, 2))                        # [dominant, multipath] per user
        Ap = [arrays.ula_steering(t * DEG, M, 0.5) for t in th]
        G = np.column_stack([arrays.spatial_signature([1.0, 0.6 * np.exp(2j * np.pi * rng.random())], A) for A in Ap])
        allpaths = np.column_stack(Ap)                                 # columns: DOM1, MULTI1, DOM2, MULTI2, ...
        W = {"Dominant DOA": np.column_stack([A[:, 0] for A in Ap]),
             "Pseudoinverse DOA": np.column_stack([arrays.pinv_weights(allpaths, 2 * l) for l in range(L)]),
             "Spatial signature": G,
             "Pseudoinverse spatial signature": np.column_stack([arrays.pinv_weights(G, l) for l in range(L)])}
        for k, Wk in W.items():
            Wk = Wk / np.linalg.norm(Wk, axis=0)
            gains[k].append(np.abs(Wk.conj().T @ G) ** 2)             # C[k, l] = |w_k^H g_l|^2
        bound.append(np.sum(np.abs(G) ** 2, axis=0))
    out = {}
    for k, C in gains.items():
        C = np.array(C)                                                 # (trials, L, L)
        sig = np.diagonal(C, axis1=1, axis2=2)
        intf = C.sum(axis=1) - sig
        out[k] = [linear_to_db(np.median(s * sig / (s * intf + 1.0))) for s in db_to_linear(snrs_db)]
    out["bound"] = [linear_to_db(np.median(s * np.array(bound))) for s in db_to_linear(snrs_db)]
    return snrs_db, out

def show_sinr(M=8, L=2):
    if 2 * L > M:
        print(f"Note: the pseudoinverse DOA method needs 2L <= M to null every path (here 2L = {2 * L} > M = {M}); "
              "with more constraints than elements it only gives a least-squares fit.")
    snrs, out = sinr_vs_snr(M, L)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for k, v in out.items():
        if k == "bound":
            ax.plot(snrs, v, "k:", label=r"bound $P\|\mathbf{g}_l\|^2/\sigma^2$")
        else:
            ax.plot(snrs, v, "o-", ms=3, label=k)
    ax.set_xlabel(r"transmit SNR $P/\sigma^2$ (dB)"); ax.set_ylabel("median per-user SINR (dB)")
    ax.set_title(f"Downlink SDMA: M={M} elements, L={L} co-channel users")
    ax.grid(alpha=0.3); ax.legend(fontsize=8); plt.show()

interact(show_sinr, M=IntSlider(value=8, min=4, max=16), L=IntSlider(value=2, min=1, max=4));

At low SNR (noise-limited) the matched beams (dominant DOA, spatial signature) win because they maximize
$|\mathbf{w}^H\mathbf{g}_l|^2$. At high SNR (interference-limited) their SINR **saturates** at the signal-to-interference
ratio, while the pseudoinverse (zero-forcing) beams remove interference and keep improving 1 dB per dB.
The pseudoinverse spatial-signature method is closest to the bound because it imposes only $L-1$ constraints.

## 8. Why array calibration matters
The algorithms above assume the measured element phases come only from the channel, i.e. that we know
$\mathbf{a}(\theta)$ exactly. Cables, connectors, oscillators and transceiver electronics add an unknown
per-element gain and phase, so the array actually measures $\mathbf{x}(t) = \mathbf{C}\,\mathbf{a}(\theta)s(t) + \mathbf{n}(t)$ with
$\mathbf{C} = \mathrm{diag}(c_1,\ldots,c_M)$. Below, $c_i = e^{j\phi_i}$ with $\phi_i \sim \mathcal{N}(0,\sigma_\phi^2)$. **Calibration** measures
$\mathbf{C}$ so it can be undone ($\mathbf{C}^{-1}\mathbf{x}$). Raise $\sigma_\phi$ and compare the uncalibrated spectra with the calibrated MUSIC reference: peaks blur, shift, and spurious peaks appear.

In [ ]:
def show_calibration(phase_err_deg=30.0, snr_db=15.0, seed=1):
    M, N = 8, 200
    doas = np.array([75, 95]) * DEG
    r = np.random.default_rng(seed)
    c = np.exp(1j * phase_err_deg * DEG * r.standard_normal(M))   # unknown per-element phase offsets
    G = arrays.ula_steering(doas, M, 0.5)
    X_true = arrays.simulate_snapshots(G, [db_to_linear(snr_db)] * 2, 1.0, N, rng=r)
    X_meas = c[:, None] * X_true                                   # what the uncalibrated array measures
    X_cal = X_meas / c[:, None]                                    # after calibration C^{-1} x
    A = arrays.ula_steering(theta_ula, M, 0.5)
    fig, ax = plt.subplots(figsize=(9, 3.8))
    curves = [("MUSIC, calibrated", arrays.music_spectrum(arrays.sample_covariance(X_cal), A, 2), dict(color="tab:gray", ls="--")),
              ("MUSIC, uncalibrated", arrays.music_spectrum(arrays.sample_covariance(X_meas), A, 2), dict(color="tab:green")),
              ("Bartlett, uncalibrated", arrays.bartlett_spectrum(arrays.sample_covariance(X_meas), A), dict(color="tab:blue"))]
    for name, Pth, style in curves:
        est = arrays.find_peaks(Pth, theta_ula, 2)
        ax.plot(theta_ula / DEG, linear_to_db(Pth / Pth.max()), **style,
                label=f"{name}: " + ", ".join(f"{e / DEG:.1f}°" for e in est))
    for t in doas: ax.axvline(t / DEG, color="k", ls=":", lw=1)
    ax.set_xlim(0, 180); ax.set_ylim(-40, 2)
    ax.set_xlabel(r"$\theta$ (deg)"); ax.set_ylabel("normalized spectrum (dB)")
    ax.set_title(fr"M={M}, per-element phase errors $\sigma_\phi$ = {phase_err_deg:.0f}° (true DOAs dotted)")
    ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower right"); plt.show()

interact(show_calibration, phase_err_deg=FloatSlider(value=30, min=0, max=90, step=2, description="σ_φ (deg)"),
         snr_db=FloatSlider(value=15, min=-5, max=40, step=1, description="SNR (dB)"),
         seed=IntSlider(value=1, min=0, max=20, description="random seed"));

## Try it yourself
* In Section 3, a ULA with $d = \lambda$ is steered to broadside. Where are the grating lobes, and why would a
  sector antenna element pattern (e.g. $120^\circ$ beamwidth facing broadside) make them harmless? What about steering to $30^\circ$?
* In Section 5, fix $\Delta\theta = 5^\circ$ and $M = 8$. Roughly how many snapshots $N$ and how much SNR does MUSIC need to
  resolve the pair reliably? Does Bartlett ever resolve it, however large $N$ or SNR get? Why?
* With the *coherent* box ticked, how many eigenvalues of $\hat{\mathbf{R}}$ rise above $\sigma^2$? Explain why MUSIC with
  $L = 2$ fails and why Bartlett does not care.
* In Section 7 with $M = 4$, increase $L$ from 1 to 4. At what point does the pseudoinverse DOA method stop working,
  and why does the pseudoinverse spatial-signature method keep working longer?